In [1]:
import os
import numpy as np
import pandas as pd
import re
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
#import statsmodels.api as sm
#from statsmodels.formula.api import ols
#from statsmodels.stats.anova import AnovaRM
#from sklearn.linear_model import LinearRegression
import csv

os.chdir('../../rf1-sra/stimuli/Scan-Investment_Game')



In [2]:
#Make a list of all the Trust Files
Trust_flist = [os.path.join(root, f) for root, dirs, files in os.walk('logs') for f in files if 'Trust-Ratings' in f]
# Make a list of Dataframes
ratings_list = []
for f in Trust_flist:
    sub = re.search('sub(.*)_', f).group(1)
    if any(i.isdigit() for i in sub):
        if len(sub)>4:
            if sub == '-10369':
                skip = 10
            elif sub == '-10478':
                skip = 10
            else:
                skip = None
            tmp_df = pd.read_csv(f,skiprows = skip)
            tmp_df['sub'] = sub[-5:]
            print(sub,tmp_df['Trait'].unique())
            ratings_list.append(tmp_df)
            

# Concatenate the DataFrames together
tpr_df = pd.concat(ratings_list)
tpr_df = tpr_df.reset_index(drop=True)

# Convert the 'Trait' column to integer type
#tpr_df['Trait'] = tpr_df['Trait'].astype(int)

# Print the updated DataFrame
print(tpr_df)
tpr_df['Trait'].unique()


#Question 1, Trait 2 for each of the 3 is "Trustworthy" in the order of friend(3), stranger(2), computer(1) 
#Question 2, Trait 1 for each of the 3 is "Likeable" in the order of friend(3), stranger(2), computer(1)
#Question 3, Trait 0 for each of the 3 is "Approachable" in the order of friend(3), stranger(2), computer(1)

-10656 [2 1 0]
-10603 [2 1 0]
-10657 [2 1 0]
-10462 [2 1 0]
-10572 [2 1 0]
-10529 [2 1 0]
-10723 [2 1 0]
-10589 [2 1 0]
-10701 [2 1 0]
-10478 [2 1 0]
-10644 [2 1 0]
-10617 [2 1 0]
-10674 [2 1 0]
-10673 [2 1 0]
-10663 [2 1 0]
-10608 [2 1 0]
-10402 [2 1 0]
-10606 [2 1 0]
-10691 [2 1 0]
-10585 [2 1 0]
-10541 [2 1 0]
-10317 [2 1 0]
-10584 [2 1 0]
-10555 [2 1 0]
-10369 [2 1 0]
-10596 [2 1 0]
-10649 [2 1 0]
-10486 [2 1 0]
-10418 [2 1 0]
-10677 [2 1 0]
     TrialNumber  Partner  Image  Trait  ran  order  Rating    sub  \
0            1.0        3      3      2  1.0    0.0     2.0  10656   
1            2.0        2      2      2  1.0    1.0     1.0  10656   
2            3.0        1      1      2  1.0    2.0    -5.0  10656   
3            4.0        3      3      1  1.0    3.0     1.0  10656   
4            5.0        2      2      1  1.0    4.0     1.0  10656   
..           ...      ...    ...    ...  ...    ...     ...    ...   
265          5.0        2      2      1  1.0    4.0     0.0 

array([2, 1, 0])

In [3]:
tpr_df[tpr_df['sub']== '-10369']

,TrialNumber,Partner,Image,Trait,ran,order,Rating,sub,﻿TrialNumber


In [4]:
tmp=pd.read_csv(f, skiprows = skip)
display(tmp)

,TrialNumber,Partner,Image,Trait,ran,order,Rating
0,1,3,3,2,1.0,0.0,0.0
1,2,2,2,2,1.0,1.0,0.0
2,3,1,1,2,1.0,2.0,0.0
3,4,3,3,1,1.0,3.0,1.0
4,5,2,2,1,1.0,4.0,0.0
5,6,1,1,1,1.0,5.0,0.0
6,7,3,3,0,1.0,6.0,0.0
7,8,2,2,0,1.0,7.0,0.0
8,9,1,1,0,1.0,8.0,-1.0


In [5]:
Trait = {2:'Trustworthy', 1:'Likeable', 0:'Approachable'}
partner = {3:'Friend',2:'Stranger',1:'Computer'}
tpr_df['partner']=tpr_df['Partner'].map(partner)
tpr_df['Outcome']=tpr_df['Trait'].map(Trait)
tpr_df
#fig = sns.barplot(y='Rating',x='Outcome',hue='partner', data=tpr_df, palette=['tab:blue','orangered','gold'], edgecolor='black', linewidth = 1)
#plt.savefig('../../derivatives/SR_partner_ratings_anova.svg')
#plt.show()

,TrialNumber,Partner,Image,Trait,ran,order,Rating,sub,﻿TrialNumber,partner,Outcome
0,1.0,3,3,2,1.0,0.0,2.0,10656,NaN,Friend,Trustworthy
1,2.0,2,2,2,1.0,1.0,1.0,10656,NaN,Stranger,Trustworthy
2,3.0,1,1,2,1.0,2.0,-5.0,10656,NaN,Computer,Trustworthy
3,4.0,3,3,1,1.0,3.0,1.0,10656,NaN,Friend,Likeable
4,5.0,2,2,1,1.0,4.0,1.0,10656,NaN,Stranger,Likeable
...,...,...,...,...,...,...,...,...,...,...,...
265,5.0,2,2,1,1.0,4.0,0.0,10677,NaN,Stranger,Likeable
266,6.0,1,1,1,1.0,5.0,0.0,10677,NaN,Computer,Likeable
267,7.0,3,3,0,1.0,6.0,0.0,10677,NaN,Friend,Approachable
268,8.0,2,2,0,1.0,7.0,0.0,10677,NaN,Stranger,Approachable


In [6]:
#This loop collapses the 6 rows of Shared Reward Partner Ratings into 1 row for each participant
collapsed = {}
for index, row in tpr_df.iterrows():
    if(not row['sub'] in collapsed):
        collapsed[row['sub']] = {}
    column_name = Trait[row['Trait']] + '-' + partner[row['Partner']]
    collapsed[row['sub']][column_name] = row['Rating']
#PR = Partner Ratings from post-scan task
PR = pd.DataFrame(collapsed).T
PR = PR.reset_index().rename(columns = {'index':'sub'})
PR

,sub,Trustworthy-Friend,Trustworthy-Stranger,Trustworthy-Computer,Likeable-Friend,Likeable-Stranger,Likeable-Computer,Approachable-Friend,Approachable-Stranger,Approachable-Computer
0,10656,2.0,1.0,-5.0,1.0,1.0,0.0,2.0,2.0,0.0
1,10603,5.0,5.0,0.0,5.0,2.0,-4.0,5.0,2.0,5.0
2,10657,4.0,-2.0,0.0,4.0,1.0,0.0,3.0,2.0,0.0
3,10462,5.0,5.0,0.0,5.0,5.0,0.0,5.0,3.0,0.0
4,10572,5.0,3.0,3.0,5.0,5.0,0.0,5.0,5.0,0.0
5,10529,4.0,2.0,3.0,5.0,3.0,0.0,4.0,4.0,0.0
6,10723,3.0,-1.0,-2.0,4.0,2.0,0.0,4.0,2.0,0.0
7,10589,-1.0,-2.0,-3.0,2.0,1.0,0.0,3.0,3.0,-2.0
8,10701,4.0,-2.0,0.0,3.0,1.0,-4.0,-2.0,2.0,5.0
9,10478,5.0,1.0,0.0,4.0,1.0,0.0,2.0,1.0,0.0
